# LeBron Shot Analytics - Visualizations

Generates all 9 charts used in the README. Each chart also saves to `analysis/charts/`.

In [ ]:
%matplotlib inline
import duckdb
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe
from matplotlib.patches import Arc, Rectangle, Circle
import numpy as np
import seaborn as sns
from pathlib import Path

ROOT = Path.cwd().parent
DB   = ROOT / 'lebron_analytics.duckdb'
OUT  = ROOT / 'analysis' / 'charts'
OUT.mkdir(parents=True, exist_ok=True)

print(f'DB : {DB}')
print(f'OUT: {OUT}')

In [ ]:
LAKERS_PURPLE = '#552583'
LAKERS_GOLD   = '#FDB927'
CAVS_WINE     = '#6F263D'
CAVS_GOLD     = '#FFB81C'
HEAT_RED      = '#98002E'
HEAT_BLACK    = '#000000'
ERA_DEAD      = '#1A6B9A'
ERA_THREE     = '#552583'
CLUTCH_RED    = '#C0392B'
REGULAR_BLUE  = '#2980B9'

TEAM_COLORS = {
	'Cavs I':  (CAVS_WINE,     CAVS_GOLD),
	'Heat':    (HEAT_RED,      '#F9A01B'),
	'Cavs II': (CAVS_WINE,     CAVS_GOLD),
	'Lakers':  (LAKERS_PURPLE, LAKERS_GOLD),
}

plt.rcParams.update({
	'font.family':        'DejaVu Sans',
	'axes.spines.top':    False,
	'axes.spines.right':  False,
	'axes.grid':          True,
	'grid.alpha':         0.3,
	'grid.linestyle':     '--',
	'figure.dpi':         150,
	'savefig.dpi':        150,
	'savefig.bbox':       'tight',
	'savefig.facecolor':  'white',
})

def connect():
	return duckdb.connect(str(DB), read_only=True)

## Chart 1 - Shot Distance Evolution

In [ ]:
def chart_distance_evolution():
	conn = connect()
	rows = conn.execute("""
		SELECT season, season_start_year, era, season_avg_shot_distance
		FROM main.mart_era_analysis
		GROUP BY season, season_start_year, era, season_avg_shot_distance
		ORDER BY season_start_year
	""").fetchall()
	conn.close()

	seasons   = [r[0] for r in rows]
	years     = [r[1] for r in rows]
	distances = [r[3] for r in rows]
	eras      = [r[2] for r in rows]

	fig, ax = plt.subplots(figsize=(13, 6))

	ax.axvspan(-0.5, 10.5, alpha=0.06, color=ERA_DEAD,  label='Dead-ball Era')
	ax.axvspan(10.5, len(seasons)-0.5, alpha=0.06, color=ERA_THREE, label='Three-point Era')
	ax.axvline(x=10.5, color='grey', linewidth=1.2, linestyle='--', alpha=0.7)
	ax.annotate('Warriors dynasty\nbegins 2014-15', xy=(10.5, 13.4),
				xytext=(12, 13.6), fontsize=8.5, color='#555',
				arrowprops=dict(arrowstyle='->', color='#888', lw=1.2))

	x = np.arange(len(seasons))
	colors = [ERA_DEAD if e == 'Dead-ball Era' else ERA_THREE for e in eras]
	for i in range(len(x) - 1):
		ax.plot(x[i:i+2], distances[i:i+2], color=colors[i+1], linewidth=2.5)
	ax.scatter(x, distances, c=colors, zorder=5, s=55, edgecolors='white', linewidths=0.8)

	for i, (xi, d, s) in enumerate(zip(x, distances, seasons)):
		if s in ('2003-04', '2012-13', '2021-22', '2023-24'):
			ax.annotate(f'{d}ft', xy=(xi, d), xytext=(xi, d + 0.28),
						ha='center', fontsize=8, fontweight='bold', color=colors[i])

	ax.set_xticks(x)
	ax.set_xticklabels([s[:4] for s in seasons], rotation=45, ha='right', fontsize=8.5)
	ax.set_ylabel('Average Shot Distance (feet)', fontsize=11)
	ax.set_title("LeBron's Shot Distance by Season", fontsize=14, fontweight='bold', pad=12)
	ax.set_ylim(9, 15)

	dead_patch  = mpatches.Patch(color=ERA_DEAD,  alpha=0.7, label='Dead-ball Era (2003-14)')
	three_patch = mpatches.Patch(color=ERA_THREE, alpha=0.7, label='Three-point Era (2014+)')
	ax.legend(handles=[dead_patch, three_patch], fontsize=9, loc='lower left')

	plt.tight_layout()
	plt.savefig(OUT / 'shot_distance_evolution.png')
	plt.show()

chart_distance_evolution()

## Chart 2 - Three-Point Transformation

In [ ]:
def chart_three_point_rate():
	conn = connect()
	rows = conn.execute("""
		SELECT era, pct_shots_that_are_3pt, pct_shots_that_are_2pt, total_shots
		FROM main.mart_era_analysis
		GROUP BY era, pct_shots_that_are_3pt, pct_shots_that_are_2pt, total_shots
		ORDER BY era
	""").fetchall()
	conn.close()

	eras   = [r[0] for r in rows]
	three  = [r[1] for r in rows]
	two    = [r[2] for r in rows]
	shots  = [r[3] for r in rows]
	colors = [ERA_DEAD, ERA_THREE]

	fig, axes = plt.subplots(1, 2, figsize=(12, 6))

	bars = axes[0].bar(eras, three, color=colors, edgecolor='white', linewidth=1.5, width=0.55)
	for bar, val in zip(bars, three):
		axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.4,
				 f'{val}%', ha='center', va='bottom', fontsize=14, fontweight='bold')
	axes[0].set_ylabel('% of Total Shots from 3-Point Range', fontsize=11)
	axes[0].set_title('3-Point Rate by Era', fontsize=13, fontweight='bold')
	axes[0].set_ylim(0, 38)
	axes[0].set_xticklabels([e.replace(' Era', '') for e in eras], fontsize=11)

	x = np.arange(len(eras))
	axes[1].bar(x, two,   color=['#2E86AB', '#7B2FBE'], label='2PT shots', width=0.5, edgecolor='white')
	axes[1].bar(x, three, bottom=two, color=['#F6A622', LAKERS_GOLD], label='3PT shots', width=0.5, edgecolor='white')
	for i, (t, th) in enumerate(zip(two, three)):
		axes[1].text(i, t/2, f'{t}%', ha='center', va='center', fontsize=13, fontweight='bold', color='white')
		axes[1].text(i, t + th/2, f'{th}%', ha='center', va='center', fontsize=13, fontweight='bold', color='white')
	axes[1].set_xticks(x)
	axes[1].set_xticklabels([e.replace(' Era', '') for e in eras], fontsize=11)
	axes[1].set_ylabel('Share of Total Shots (%)', fontsize=11)
	axes[1].set_title('Shot Type Mix by Era', fontsize=13, fontweight='bold')
	axes[1].set_ylim(0, 115)
	axes[1].legend(fontsize=10)
	for i, s in enumerate(shots):
		axes[1].text(i, 105, f'n={s:,}', ha='center', fontsize=9, color='#555')

	plt.suptitle('The Three-Point Transformation', fontsize=15, fontweight='bold', y=1.01)
	plt.tight_layout()
	plt.savefig(OUT / 'three_point_rate.png')
	plt.show()

chart_three_point_rate()

## Chart 3 - FG% by Tenure

In [ ]:
def chart_fg_by_tenure():
	conn = connect()
	rows = conn.execute("""
		SELECT tenure, tenure_order,
			   MIN(first_season) as first_s, MAX(last_season) as last_s,
			   SUM(total_shots) as shots,
			   ROUND(SUM(total_made)*100.0/SUM(total_shots),1) as fg_pct,
			   ROUND(SUM(shots_3pt)*100.0/SUM(total_shots),1) as three_rate
		FROM (
			SELECT tenure, tenure_order, first_season, last_season,
				   total_shots, total_made, shots_3pt
			FROM main.mart_tenure_performance
			GROUP BY tenure, tenure_order, first_season, last_season,
					 total_shots, total_made, shots_3pt
		)
		GROUP BY tenure, tenure_order ORDER BY tenure_order
	""").fetchall()
	conn.close()

	tenures     = [r[0] for r in rows]
	fg_pcts     = [r[5] for r in rows]
	three_rates = [r[6] for r in rows]
	shot_cnts   = [r[4] for r in rows]
	seasons     = [f"{r[2][:4]}-{r[3][:4]}" for r in rows]
	face_clrs   = [TEAM_COLORS[t][0] for t in tenures]
	edge_clrs   = [TEAM_COLORS[t][1] for t in tenures]

	fig, ax = plt.subplots(figsize=(11, 7))
	ax.axhline(y=46.0, color='#888', linewidth=1.3, linestyle='--', alpha=0.7, label='~NBA avg FG% (~46%)')

	x = np.arange(len(tenures))
	bars = ax.bar(x, fg_pcts, color=face_clrs, edgecolor=edge_clrs, linewidth=2.5, width=0.55, zorder=3)

	for bar, fg, shots, seas, t3 in zip(bars, fg_pcts, shot_cnts, seasons, three_rates):
		ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.25,
				f'{fg}%', ha='center', va='bottom', fontsize=16, fontweight='bold',
				color=bar.get_facecolor())
		ax.text(bar.get_x() + bar.get_width()/2, bar.get_height()/2,
				f'{shots:,} shots\n{seas}\n3PT: {t3}%',
				ha='center', va='center', fontsize=8.5, color='white', fontweight='bold',
				multialignment='center')

	ax.set_xticks(x)
	ax.set_xticklabels(tenures, fontsize=12, fontweight='bold')
	ax.set_ylabel('Field Goal Percentage', fontsize=12)
	ax.set_title("LeBron's FG% Across Four Team Tenures", fontsize=14, fontweight='bold', pad=12)
	ax.set_ylim(43, 58)
	ax.legend(fontsize=10)
	ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0f}%'))

	plt.tight_layout()
	plt.savefig(OUT / 'fg_by_tenure.png')
	plt.show()

chart_fg_by_tenure()

## Chart 4 - Clutch Performance

In [ ]:
def chart_clutch_comparison():
	conn = connect()
	rows = conn.execute("""
		SELECT situation, fg_pct, three_pt_rate, points_per_shot, total_shots,
			   critical_fg_pct, final_possession_fg_pct
		FROM main.mart_clutch_gene
		ORDER BY is_clutch_time DESC
	""").fetchall()
	conn.close()

	clutch_row  = rows[0]
	regular_row = rows[1]

	categories   = ['Overall FG%', '3PT Rate', 'Points/Shot x 20']
	clutch_vals  = [clutch_row[1],  clutch_row[2],  clutch_row[3] * 20]
	regular_vals = [regular_row[1], regular_row[2], regular_row[3] * 20]

	fig, axes = plt.subplots(1, 2, figsize=(13, 6))

	x = np.arange(len(categories))
	w = 0.35
	b1 = axes[0].bar(x - w/2, regular_vals, w, color=REGULAR_BLUE, label='Regular', edgecolor='white', linewidth=1.5)
	b2 = axes[0].bar(x + w/2, clutch_vals,  w, color=CLUTCH_RED,   label='Clutch',  edgecolor='white', linewidth=1.5)

	def label_bars(bars, vals):
		for bar, v in zip(bars, vals):
			disp = f'{v/20:.3f}' if bars == b1 and v == vals[-1] else f'{v:.1f}%'
			axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
						 disp, ha='center', va='bottom', fontsize=10, fontweight='bold')

	label_bars(b1, regular_vals)
	label_bars(b2, clutch_vals)
	axes[0].set_xticks(x)
	axes[0].set_xticklabels(categories, fontsize=10)
	axes[0].set_ylabel('Value', fontsize=11)
	axes[0].set_title('Regular vs Clutch Situations', fontsize=13, fontweight='bold')
	axes[0].legend(fontsize=10)
	axes[0].set_ylim(0, 58)

	tenures      = ['Cavs I', 'Heat', 'Cavs II', 'Lakers']
	clutch_by_t  = [45.3, 48.7, 50.8, 47.0]
	regular_by_t = [47.4, 54.3, 52.6, 51.3]
	t_colors     = [TEAM_COLORS[t][0] for t in tenures]

	x2 = np.arange(len(tenures))
	axes[1].bar(x2 - 0.2, regular_by_t, 0.38, color=t_colors, alpha=0.45, edgecolor='grey', linewidth=1, label='Overall FG%')
	axes[1].bar(x2 + 0.2, clutch_by_t,  0.38, color=t_colors, edgecolor='grey', linewidth=1, label='Clutch FG%')

	for xi, cv, rv in zip(x2, clutch_by_t, regular_by_t):
		axes[1].text(xi + 0.2, cv + 0.3, f'{cv}%', ha='center', fontsize=9.5, fontweight='bold')
		axes[1].text(xi - 0.2, rv + 0.3, f'{rv}%', ha='center', fontsize=9.5)

	axes[1].set_xticks(x2)
	axes[1].set_xticklabels(tenures, fontsize=10.5)
	axes[1].set_ylabel('Field Goal Percentage', fontsize=11)
	axes[1].set_title('Clutch vs Overall FG% by Tenure', fontsize=13, fontweight='bold')
	axes[1].legend(fontsize=9)
	axes[1].set_ylim(42, 60)
	axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0f}%'))

	plt.suptitle('The Clutch Gene: Pressure Performance', fontsize=15, fontweight='bold', y=1.01)
	plt.tight_layout()
	plt.savefig(OUT / 'clutch_comparison.png')
	plt.show()

chart_clutch_comparison()

## Chart 5 - Shot Zone Distribution

In [ ]:
def chart_shot_distribution():
	conn = connect()
	rows = conn.execute("""
		SELECT era,
			   ROUND(shots_at_rim*100.0/total_shots,1),
			   ROUND(shots_close_range*100.0/total_shots,1),
			   ROUND(shots_short_midrange*100.0/total_shots,1),
			   ROUND(shots_midrange*100.0/total_shots,1),
			   ROUND(shots_long_midrange*100.0/total_shots,1),
			   ROUND(shots_threepoint*100.0/total_shots,1)
		FROM main.mart_era_analysis
		GROUP BY era, shots_at_rim, total_shots, shots_close_range,
				 shots_short_midrange, shots_midrange, shots_long_midrange, shots_threepoint
		ORDER BY era
	""").fetchall()
	conn.close()

	zones     = ['At Rim', 'Close Range', 'Short Mid', 'Mid-range', 'Long Mid', '3-Point']
	zone_clrs = ['#1A6B9A', '#2196F3', '#64B5F6', '#FFC107', '#FF7043', '#7B1FA2']
	eras      = [r[0] for r in rows]
	data      = [[r[i+1] for i in range(6)] for r in rows]

	fig, ax = plt.subplots(figsize=(11, 6))
	x   = np.arange(len(eras))
	w   = 0.52
	cum = np.zeros(len(eras))

	for zi, (zone, clr) in enumerate(zip(zones, zone_clrs)):
		vals = np.array([d[zi] for d in data])
		bars = ax.bar(x, vals, w, bottom=cum, color=clr, label=zone, edgecolor='white', linewidth=0.8)
		for xi, (v, b) in enumerate(zip(vals, cum)):
			if v >= 5.0:
				ax.text(xi, b + v/2, f'{v}%', ha='center', va='center',
						fontsize=10, fontweight='bold', color='white')
		cum += vals

	ax.set_xticks(x)
	ax.set_xticklabels([e.replace(' Era', '') for e in eras], fontsize=12, fontweight='bold')
	ax.set_ylabel('Share of Total Shots (%)', fontsize=11)
	ax.set_title('Shot Zone Distribution by Era', fontsize=14, fontweight='bold', pad=12)
	ax.legend(loc='upper right', ncol=2, fontsize=9.5, framealpha=0.9)
	ax.set_ylim(0, 115)
	ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0f}%'))

	dead_vals  = data[0]
	three_vals = data[1]
	changes = [(i, three_vals[i] - dead_vals[i]) for i in range(6)]
	bigg = max(changes, key=lambda c: abs(c[1]))
	ax.annotate(f'{zones[bigg[0]]}\n{bigg[1]:+.1f}pp change', xy=(1, 50),
				fontsize=9, color='#333', style='italic')

	plt.tight_layout()
	plt.savefig(OUT / 'shot_distribution.png')
	plt.show()

chart_shot_distribution()

## Chart 6 - Top Opponents

In [ ]:
def chart_top_opponents():
	conn = connect()
	rows = conn.execute("""
		SELECT opponent_abbr, total_shots, fg_pct, clutch_fg_pct, games_played
		FROM main.mart_opponent_analysis
		GROUP BY opponent_abbr, total_shots, fg_pct, clutch_fg_pct, games_played
		ORDER BY total_shots DESC LIMIT 10
	""").fetchall()
	conn.close()

	opps  = [r[0] for r in reversed(rows)]
	shots = [r[1] for r in reversed(rows)]
	fgs   = [r[2] for r in reversed(rows)]
	cfgs  = [r[3] for r in reversed(rows)]
	games = [r[4] for r in reversed(rows)]

	norm   = plt.Normalize(min(fgs), max(fgs))
	colors = plt.cm.RdYlGn(norm(fgs))

	fig, ax = plt.subplots(figsize=(11, 8))
	y    = np.arange(len(opps))
	bars = ax.barh(y, shots, color=colors, edgecolor='white', linewidth=1.2, height=0.6)

	for i, (bar, fg, cfg, g) in enumerate(zip(bars, fgs, cfgs, games)):
		cfg_str = f'{cfg}%' if cfg else 'N/A'
		ax.text(bar.get_width() + 12, i,
				f'FG: {fg}%  |  Clutch: {cfg_str}  |  {g} games',
				va='center', fontsize=8.5, color='#333')
		ax.text(bar.get_width() - 30, i, f'{shots[i]:,}',
				va='center', ha='right', fontsize=9, fontweight='bold', color='white')

	ax.set_yticks(y)
	ax.set_yticklabels(opps, fontsize=12, fontweight='bold')
	ax.set_xlabel('Total Shot Attempts', fontsize=11)
	ax.set_title('Most-Faced Opponents (career shot attempts)', fontsize=14, fontweight='bold', pad=12)
	ax.set_xlim(0, max(shots) + 280)

	sm = plt.cm.ScalarMappable(cmap='RdYlGn', norm=norm)
	sm.set_array([])
	cbar = plt.colorbar(sm, ax=ax, shrink=0.6, pad=0.01)
	cbar.set_label('FG% (green = more efficient)', fontsize=9)

	plt.tight_layout()
	plt.savefig(OUT / 'top_opponents.png')
	plt.show()

chart_top_opponents()

## Chart 7 - Shot Chart Heatmap

In [ ]:
def draw_court(ax, color='#333333', lw=1.5):
	ax.set_facecolor('#F8F0E3')
	basket    = Circle((0, 0), 7.5, linewidth=lw, color=color, fill=False)
	backboard = plt.Line2D([-30, 30], [-7.5, -7.5], linewidth=lw, color=color)
	ax.add_patch(basket)
	ax.add_line(backboard)
	paint_outer = Rectangle((-80, -47.5), 160, 190, linewidth=lw, color=color, fill=False)
	paint_inner = Rectangle((-60, -47.5), 120, 190, linewidth=lw, color=color, fill=False)
	ax.add_patch(paint_outer)
	ax.add_patch(paint_inner)
	ft_top = Arc((0, 142.5), 120, 120, theta1=0,   theta2=180, linewidth=lw, color=color)
	ft_bot = Arc((0, 142.5), 120, 120, theta1=180, theta2=0,   linewidth=lw, color=color, linestyle='dashed')
	ax.add_patch(ft_top)
	ax.add_patch(ft_bot)
	restricted = Arc((0, 0), 80, 80, theta1=0, theta2=180, linewidth=lw, color=color)
	ax.add_patch(restricted)
	corner3_left  = plt.Line2D([-220, -220], [-47.5, 92.5], linewidth=lw, color=color)
	corner3_right = plt.Line2D([220, 220],   [-47.5, 92.5], linewidth=lw, color=color)
	three_arc     = Arc((0, 0), 475, 475, theta1=22, theta2=158, linewidth=lw, color=color)
	ax.add_line(corner3_left)
	ax.add_line(corner3_right)
	ax.add_patch(three_arc)
	mid_circle_out = Circle((0, 422.5), 60, linewidth=lw, color=color, fill=False)
	mid_circle_in  = Circle((0, 422.5), 20, linewidth=lw, color=color, fill=False)
	half_line      = plt.Line2D([-250, 250], [422.5, 422.5], linewidth=lw, color=color)
	ax.add_patch(mid_circle_out)
	ax.add_patch(mid_circle_in)
	ax.add_line(half_line)
	outer = Rectangle((-250, -47.5), 500, 470, linewidth=lw+0.5, color=color, fill=False)
	ax.add_patch(outer)
	ax.set_xlim(-250, 250)
	ax.set_ylim(-47.5, 422.5)
	ax.set_aspect('equal')
	ax.set_xticks([])
	ax.set_yticks([])


def chart_shot_heatmap():
	conn = connect()
	shots = conn.execute("""
		SELECT loc_x, loc_y, is_shot_made::int as made
		FROM main.stg_lebron_shots
		WHERE loc_x IS NOT NULL AND loc_y IS NOT NULL
		  AND loc_y <= 422.5
	""").fetchall()
	conn.close()

	lx   = np.array([s[0] for s in shots], dtype=float)
	ly   = np.array([s[1] for s in shots], dtype=float)
	made = np.array([s[2] for s in shots], dtype=float)

	fig, axes = plt.subplots(1, 2, figsize=(15, 9))

	draw_court(axes[0])
	hb = axes[0].hexbin(lx, ly, gridsize=35, mincnt=1,
						cmap='hot_r', bins='log', extent=(-250, 250, -47.5, 422.5))
	plt.colorbar(hb, ax=axes[0], label='Shot frequency (log scale)', shrink=0.7)
	axes[0].set_title('All Shot Attempts - Frequency Density', fontsize=12, fontweight='bold', pad=8)

	draw_court(axes[1])
	miss_mask = made == 0
	made_mask = made == 1
	axes[1].scatter(lx[miss_mask], ly[miss_mask], c='#E74C3C', alpha=0.22, s=6, label='Miss', rasterized=True)
	axes[1].scatter(lx[made_mask], ly[made_mask], c='#2ECC71', alpha=0.30, s=6, label='Made', rasterized=True)
	axes[1].legend(markerscale=3, fontsize=10, loc='upper right')
	axes[1].set_title('Made vs Missed - Career', fontsize=12, fontweight='bold', pad=8)

	fig.suptitle('LeBron James - Career Shot Chart (2003-2024)', fontsize=15, fontweight='bold', y=1.00)
	plt.tight_layout()
	plt.savefig(OUT / 'shot_chart_heatmap.png', dpi=150)
	plt.show()

chart_shot_heatmap()

## Chart 8 - Career FG% Trend

In [ ]:
def chart_career_fg_trend():
	conn = connect()
	rows = conn.execute("""
		SELECT season, season_start_year, era, season_fg_pct, season_shots
		FROM main.mart_era_analysis
		GROUP BY season, season_start_year, era, season_fg_pct, season_shots
		ORDER BY season_start_year
	""").fetchall()
	conn.close()

	seasons = [r[0] for r in rows]
	eras    = [r[2] for r in rows]
	fgs     = [r[3] for r in rows]

	fig, ax = plt.subplots(figsize=(13, 6))
	ax.axvspan(-0.5, 10.5, alpha=0.06, color=ERA_DEAD)
	ax.axvspan(10.5, len(seasons)-0.5, alpha=0.06, color=ERA_THREE)
	ax.axvline(x=10.5, color='grey', linewidth=1.2, linestyle='--', alpha=0.7)

	champ_seasons = {'2011-12', '2012-13', '2015-16', '2019-20'}
	x    = np.arange(len(seasons))
	clrs = [ERA_DEAD if e == 'Dead-ball Era' else ERA_THREE for e in eras]

	for i in range(len(x) - 1):
		ax.plot(x[i:i+2], fgs[i:i+2], color=clrs[i], linewidth=2.5, alpha=0.9)
	ax.scatter(x, fgs, c=clrs, zorder=5, s=60, edgecolors='white', linewidths=0.8)

	for s, xi, fg in zip(seasons, x, fgs):
		if s in champ_seasons:
			ax.annotate('NBA\nChamp', xy=(xi, fg), xytext=(xi + 0.15, fg + 1.5),
						fontsize=7.5, color=LAKERS_GOLD, fontweight='bold',
						arrowprops=dict(arrowstyle='->', color=LAKERS_GOLD, lw=1))
			ax.scatter([xi], [fg], c=LAKERS_GOLD, zorder=6, s=100, edgecolors='black', linewidths=0.8)

	z  = np.polyfit(x, fgs, 2)
	p  = np.poly1d(z)
	xn = np.linspace(0, len(x)-1, 200)
	ax.plot(xn, p(xn), '--', color='#555', linewidth=1.5, alpha=0.6, label='Trend (quadratic fit)')

	peak_i = int(np.argmax(fgs))
	low_i  = int(np.argmin(fgs))
	ax.annotate(f'Career high\n{fgs[peak_i]}%', xy=(x[peak_i], fgs[peak_i]),
				xytext=(x[peak_i]-1.8, fgs[peak_i]+0.8), fontsize=8.5, color='green',
				fontweight='bold', arrowprops=dict(arrowstyle='->', color='green', lw=1))
	ax.annotate(f'Rookie season\n{fgs[low_i]}%', xy=(x[low_i], fgs[low_i]),
				xytext=(x[low_i]+0.5, fgs[low_i]-1.6), fontsize=8.5, color='red',
				arrowprops=dict(arrowstyle='->', color='red', lw=1))

	ax.set_xticks(x)
	ax.set_xticklabels([s[:4] for s in seasons], rotation=45, ha='right', fontsize=8.5)
	ax.set_ylabel('Field Goal Percentage (%)', fontsize=11)
	ax.set_title("LeBron's Career FG% - Season by Season", fontsize=14, fontweight='bold', pad=12)
	ax.set_ylim(38, 60)
	ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0f}%'))

	dead_patch  = mpatches.Patch(color=ERA_DEAD,  alpha=0.7, label='Dead-ball Era')
	three_patch = mpatches.Patch(color=ERA_THREE, alpha=0.7, label='Three-point Era')
	trend_line  = plt.Line2D([0], [0], color='#555', linewidth=1.5, linestyle='--', label='Trend')
	ax.legend(handles=[dead_patch, three_patch, trend_line], fontsize=9, loc='upper left')

	plt.tight_layout()
	plt.savefig(OUT / 'career_fg_trend.png')
	plt.show()

chart_career_fg_trend()

## Chart 9 - Best Individual Seasons

In [ ]:
def chart_best_seasons():
	conn = connect()
	rows = conn.execute("""
		SELECT season, tenure, total_shots, total_points, fg_pct, points_per_shot,
			   rank_by_shots, rank_by_points, rank_by_fg_pct, rank_by_pps
		FROM main.mart_best_seasons
		ORDER BY season_start_year
	""").fetchall()
	conn.close()

	tenure_face = {'Cavs I': CAVS_WINE, 'Heat': HEAT_RED, 'Cavs II': CAVS_WINE, 'Lakers': LAKERS_PURPLE}
	tenure_edge = {'Cavs I': CAVS_GOLD, 'Heat': '#F9A01B', 'Cavs II': CAVS_GOLD, 'Lakers': LAKERS_GOLD}

	by_shots = sorted(rows, key=lambda r: r[6])[:8]
	by_pts   = sorted(rows, key=lambda r: r[7])[:8]
	by_eff   = sorted(rows, key=lambda r: r[8])[:8]

	fig, axes = plt.subplots(1, 3, figsize=(17, 7))

	def hbar(ax, data, val_idx, xlabel, title, fmt='{:.0f}'):
		labels  = [r[0] for r in reversed(data)]
		vals    = [r[val_idx] for r in reversed(data)]
		tenures = [r[1] for r in reversed(data)]
		colors  = [tenure_face.get(t, '#888') for t in tenures]
		edges   = [tenure_edge.get(t, '#aaa') for t in tenures]
		y       = np.arange(len(labels))
		bars    = ax.barh(y, vals, color=colors, edgecolor=edges, linewidth=1.6, height=0.58)
		for bar, v in zip(bars, vals):
			ax.text(bar.get_width() - (max(vals) * 0.03), bar.get_y() + bar.get_height()/2,
					fmt.format(v), va='center', ha='right', fontsize=9, fontweight='bold', color='white')
		ax.set_yticks(y)
		ax.set_yticklabels(labels, fontsize=9.5)
		ax.set_xlabel(xlabel, fontsize=10)
		ax.set_title(title, fontsize=12, fontweight='bold', pad=8)

	hbar(axes[0], by_shots, 2, 'Shot Attempts',    'Top 8 - Volume')
	hbar(axes[1], by_pts,   3, 'Field-Goal Points', 'Top 8 - Scoring')
	hbar(axes[2], by_eff,   4, 'Field Goal %',      'Top 8 - Effectiveness', fmt='{:.1f}%')

	legend_handles = [
		mpatches.Patch(facecolor=CAVS_WINE,     edgecolor=CAVS_GOLD,   label='Cavs I / Cavs II'),
		mpatches.Patch(facecolor=HEAT_RED,      edgecolor='#F9A01B',   label='Heat'),
		mpatches.Patch(facecolor=LAKERS_PURPLE, edgecolor=LAKERS_GOLD, label='Lakers'),
	]
	fig.legend(handles=legend_handles, loc='lower center', ncol=3, fontsize=9.5,
			   framealpha=0.9, bbox_to_anchor=(0.5, -0.04))

	plt.suptitle('LeBron James - Best Individual Seasons', fontsize=15, fontweight='bold', y=1.01)
	plt.tight_layout()
	plt.savefig(OUT / 'best_seasons.png', bbox_inches='tight')
	plt.show()

chart_best_seasons()